In [0]:
%sql
-- STEP 1 — BRONZE → SILVER: unpack raw Adzuna JSON into typed columns
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TABLE silver.job_postings AS
SELECT
  raw_json:id                    AS job_id,
  raw_json:title                 AS title,
  raw_json:company.display_name  AS company,
  raw_json:location.display_name AS location,
  raw_json:category.label        AS category,
  raw_json:contract_time         AS contract_time,
  raw_json:salary_min::double    AS salary_min,
  raw_json:salary_max::double    AS salary_max,
  'USD'                          AS salary_currency,
  raw_json:description           AS description,
  raw_json:created::timestamp    AS created,
  ingestion_date,
  ingestion_ts,
  source
FROM bronze.adzuna_jobs_raw

In [0]:
df = spark.table("silver.job_postings")

print("rows:", df.count())
df.printSchema()
df.show(5, truncate=60)

In [0]:
df.count()

In [0]:
df.select("job_id").distinct().count()

In [0]:
from pyspark.sql import functions as F
df.select([
    F.sum(
        F.when(F.col(col_name).isNull(), 1).otherwise(0)
    ).alias(col_name)
    for col_name in df.columns
]).display()

In [0]:
from pyspark.sql import functions as F

# 1. split: clean rows vs broken rows (missing a must-have field)
good = df.filter(F.col("job_id").isNotNull() & F.col("title").isNotNull())
bad  = df.filter(F.col("job_id").isNull()  | F.col("title").isNull())

# 2. fill the contract_time blanks on the good rows
good = good.fillna({"contract_time": "unknown"})

# 3. see the split
good.count()   # run this
bad.count()    # and this

In [0]:
good.show(5, truncate=60)

In [0]:
bad.writeTo("silver.quarantine_jobs").createOrReplace()

In [0]:
good.writeTo("silver.job_postings").createOrReplace()

In [0]:
jobs = spark.table("silver.job_postings")

In [0]:
from pyspark.sql import functions as F

skills = [
    # languages
    "python", "java", "javascript", "typescript", "scala", "sql", "cobol", "c#", "php", "ruby",
    # data / engineering
    "spark", "hadoop", "kafka", "airflow", "snowflake", "databricks", "etl",
    "redshift", "bigquery", "hive", "mongodb", "postgres", "mysql", "oracle", "nosql",
    # cloud
    "aws", "azure", "gcp",
    # devops
    "docker", "kubernetes", "terraform", "jenkins", "git", "linux", "ansible",
    # bi / analytics
    "tableau", "power bi", "excel", "looker", "analytics",
    # concepts / roles
    "machine learning", "ml", "ai", "viz", "nlp", "agile", "scrum", "data", "cloud",
]
jobs = jobs.withColumn("search_text",
          F.lower(F.concat_ws(" ", F.col("title"), F.col("description"))))

jobs.select("job_id", "title", "search_text").show(3, truncate=80)

In [0]:
import re
from pyspark.sql import functions as F

# for each skill, check if it appears as a whole word; if yes, keep the skill name
matched = F.array(*[
    F.when(F.col("search_text").rlike(r"\b" + re.escape(s) + r"\b"), F.lit(s))
    for s in skills
])

# drop the misses (nulls), leaving only the skills actually found
jobs = jobs.withColumn("skills_arr", F.array_compact(matched))

jobs.select("job_id", "title", "skills_arr").show(10, truncate=60)

In [0]:
# persist the enriched data (all fields + skills_arr, without the helper search_text)
jobs.drop("search_text").writeTo("silver.job_postings_skills").createOrReplace()

print("rows written:", spark.table("silver.job_postings_skills").count())

In [0]:
from pyspark.sql import functions as F

# ---- data modeling: derived columns + combined numerics ----
modelled = (spark.table("silver.job_postings_skills")
    .withColumn("salary_mid", F.round((F.col("salary_min") + F.col("salary_max")) / 2, 2))
    .withColumn("salary_range", F.round(F.col("salary_max") - F.col("salary_min"), 2))
    .withColumn("has_salary", F.col("salary_mid").isNotNull())
    .withColumn("skills_count", F.size("skills_arr"))
    .withColumn("posting_age_days", F.datediff("ingestion_date", "created"))
    .withColumn("location_city", F.split("location", ",").getItem(0))
    .withColumn("location_region", F.element_at(F.split("location", ","), -1)))

modelled.writeTo("silver.job_postings_skills").createOrReplace()

print("rows:", modelled.count(), "| columns:", len(modelled.columns))
modelled.select("job_id", "title", "location_city", "location_region",
                "salary_min", "salary_max", "salary_mid", "salary_range",
                "skills_count", "posting_age_days").show(5, truncate=40)